# Mission 7: Queue Dynamics — Trading the Realistic Exchange (L3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_07_queue_dynamics/notebook.ipynb)

**Advanced elective.** Mission 2 traded an *aggregated* (L2) book, where you see depth at each price.
Real exchanges are **order-by-order (L3)**: every resting order has an identity and a place in a
first-in-first-out (FIFO) queue at its price. Whether your passive order makes money depends on a
question L2 can't even ask — *where are you in the queue, and will the market move before you reach
the front?*

**Learning objectives**
- Explain FIFO queue priority and why **queue position** is the maker's core asset
- Simulate a resting limit order order-by-order: drain the queue ahead, then fill
- Model the **cancel race**: a maker pulling a quote against latency, and **adverse selection**
- Observe queue dynamics on a **real Bitstamp BTC/USD L3 feed**

---

## Background

A *passive* (limit) order doesn't trade immediately — it joins the back of the queue at its price
and waits. Orders ahead of it must leave first (by trading or cancelling) before it reaches the
front. Only then does the next opposing market order fill it.

This creates a tension every market maker lives with:

- You want to **rest** to earn the spread (and often a maker rebate).
- But if information arrives — the fair price moves against your quote — you'd rather **cancel** than
  get filled at a now-stale price. That's **adverse selection**.
- Cancelling takes time (**latency**). If a trade reaches you inside that window, you're filled
  anyway — adversely.

We'll build all of this from first principles, then watch it happen in real data.

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Pinned to the exact commit so the L3 engine (mbo) matches this lesson.
    !pip install -q "git+https://github.com/convexpi/arena.git@8af48e512e15a29a4b398b70a7ca9d9e812a2e04"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from convexpi.arena.mbo import L3Book, simulate_passive_order, load_l3

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
print('Ready.')

---
## Part 1: The L3 mental model

An L3 event stream is a sequence of **order** events (`created` / `deleted`) and **trade** events.
Each order has an `id`, a `price` (`p`), a remaining `amount` (`a`), a `side` (`s`: 0 = buy/bid,
1 = sell/ask) and a microsecond timestamp (`t`). The `L3Book` replays these into a live book where
each price level holds a **FIFO list of order ids**.

Let's build a tiny book by hand and look at the queue at a single price.

In [ ]:
book = L3Book()
# Three makers post bids of 1.0 BTC each at $60,000 — in this order.
for oid in (101, 102, 103):
    book.apply({"k": "o", "e": "created", "id": oid, "p": 60000.0, "a": 1.0, "s": 0, "t": oid})

print("best bid / ask:", book.best_bid(), "/", book.best_ask())
print("size resting at $60,000 bid:", book.size_at(0, 60000.0), "BTC")
print("queue order (front -> back):", book.order_ids_at(0, 60000.0))

# Order 101 cancels. The queue closes up; 102 is now at the front.
book.apply({"k": "o", "e": "deleted", "id": 101, "p": 60000.0, "a": 1.0, "s": 0, "t": 200})
print("after 101 cancels       :", book.order_ids_at(0, 60000.0), "->", book.size_at(0, 60000.0), "BTC")

The key point: **a price level is not a number, it's a line of people.** If you join the bid at
\$60,000 with 2 BTC already resting, you are *behind* 2 BTC. Every one of those 2 BTC must trade or
cancel before a single satoshi of yours fills. That's queue position.

---
## Part 2: Queue position to fill

`simulate_passive_order` rests an order of `size` at `price` (side 0 = buy, 1 = sell) at event index
`enter_idx`, then replays the stream order-by-order, tracking how much of the queue is **ahead** of
us until we either fill or cancel.

We construct a clean scenario: three makers (3 BTC total) rest ahead of us; then a series of *sell*
market orders (taker side 1) hit the bid, draining the queue and eventually filling our buy.

In [ ]:
P = 60000.0
def build_stream():
    ev, t = [], 0
    for oid in (1, 2, 3):                       # 3 BTC resting ahead of us on the bid
        ev.append({"k": "o", "e": "created", "id": oid, "p": P, "a": 1.0, "s": 0, "t": t}); t += 1_000_000
    enter = len(ev)                             # we join HERE -> 3.0 BTC ahead
    ev.append({"k": "o", "e": "created", "id": 9, "p": P - 1, "a": 5.0, "s": 0, "t": t}); t += 1_000_000
    for _ in range(5):                          # sell market orders hit the bid, 1 BTC each, every 2s
        ev.append({"k": "t", "p": P, "a": 1.0, "s": 1, "t": t}); t += 2_000_000
    return ev, enter

ev, enter = build_stream()
r = simulate_passive_order(ev, side=0, price=P, enter_idx=enter, size=0.5)
print(f"queue ahead at entry : {r.initial_queue_ahead:.2f} BTC")
print(f"filled               : {r.filled}")
print(f"time to fill         : {r.time_to_fill_s:.1f} s")
print(f"reached front after  : {(r.reached_front_ts - r.enter_ts)/1e6:.1f} s")

In [ ]:
# Visualise the queue ahead of us draining to zero, then our fill.
ts = np.array([(t - r.enter_ts) / 1e6 for t, q in r.queue_trace])
qa = np.array([q for t, q in r.queue_trace])
plt.figure(figsize=(8, 3.2))
plt.step(ts, qa, where='post', lw=2)
plt.axhline(0, color='k', lw=0.8)
if r.fill_ts:
    plt.scatter([(r.fill_ts - r.enter_ts)/1e6], [0], color='crimson', zorder=5, label='filled')
plt.xlabel('seconds since we joined the queue'); plt.ylabel('BTC ahead of us')
plt.title('Queue position draining to a fill'); plt.legend(); plt.tight_layout(); plt.show()

**Exercise 2.1** — You rested 0.5 BTC behind 3.0 BTC, and each market order was 1 BTC arriving every
2 s. Predict the fill time *before* running, then change `size` to 2.0 and re-run. Why does a larger
order take no longer to *start* filling but you should think about partial fills?

---
## Part 3: The cancel race & adverse selection

Now the maker's dilemma. You post a quote, but after holding it for `cancel_after_s` you decide the
market has moved and you want out. Your cancel takes `latency_us` microseconds to reach the exchange.
Two outcomes:

- **Fast cancel** — your cancel lands before any trade reaches you. You escape clean.
- **Slow cancel** — a trade reaches the front and fills you *inside* the latency window. You were
  **adversely selected**: filled at a price you no longer wanted.

In [ ]:
ev, enter = build_stream()
base = simulate_passive_order(ev, side=0, price=P, enter_idx=enter, size=0.5)
fast = simulate_passive_order(ev, side=0, price=P, enter_idx=enter, size=0.5,
                              cancel_after_s=1.0, latency_us=100_000)     # decide @1s, lands +0.1s
slow = simulate_passive_order(ev, side=0, price=P, enter_idx=enter, size=0.5,
                              cancel_after_s=7.0, latency_us=5_000_000)   # decide @7s, lands +5s

for name, res in [("no cancel ", base), ("fast cancel", fast), ("slow cancel", slow)]:
    outcome = "FILLED" if res.filled else ("cancelled" if res.cancelled else "open")
    print(f"{name}: {outcome:10}  (filled={res.filled}, cancelled={res.cancelled})")

The **fast cancel escapes**; the **slow cancel gets filled before the cancel lands**. Same quote,
same market — only the latency changed. This is why low latency is worth so much to market makers:
not (only) to be first in the queue, but to *get out in time*. The adverse-selection cost of being
slow is a direct, modellable PnL leak.

**Exercise 3.1** — Hold `cancel_after_s=2.0` fixed and sweep `latency_us` over
`[50_000, 500_000, 2_000_000, 6_000_000]`. Find the latency at which the outcome flips from
*cancelled* to *filled*. That threshold is your adverse-selection boundary for this scenario.

---
## Part 4: Real data — a Bitstamp BTC/USD L3 feed

Constructed streams make the mechanics obvious; real markets are messier. Here we load ~75 seconds of
a **real order-by-order Bitstamp feed** (every add, cancel, and trade) and reconstruct the book.

> Note: this is a public capture with no opening snapshot, so the very top of the book can look
> momentarily *crossed* during warm-up — a real artifact of L3 reconstruction. We sample well past
> the start and skip crossed touches.

In [ ]:
import urllib.request
URL = ("https://raw.githubusercontent.com/convexpi/arena/8af48e512e15a29a4b398b70a7ca9d9e812a2e04/"
       "data/btcusd_l3_sample.jsonl")
urllib.request.urlretrieve(URL, "btcusd_l3_sample.jsonl")
events = load_l3("btcusd_l3_sample.jsonl")
span_s = (events[-1]['t'] - events[0]['t']) / 1e6
n_trades = sum(1 for e in events if e['k'] == 't')
print(f"{len(events):,} events over {span_s:.1f} s  ({n_trades:,} trades)")

In [ ]:
# Sample many entry points; at each, rest a small buy at the current best bid and simulate to fill.
book = L3Book(); samples = []
for i, e in enumerate(events):
    if e['k'] == 'o':
        book.apply(e)
    if i > 800 and i % 150 == 0:
        bb, ba = book.best_bid(), book.best_ask()
        if bb is not None and ba is not None and bb < ba:    # skip crossed touches
            samples.append((i, bb))

ttfs = []
for i, bb in samples:
    r = simulate_passive_order(events, side=0, price=bb, enter_idx=i, size=0.02)
    if r.filled:
        ttfs.append(r.time_to_fill_s)
print(f"clean entry points        : {len(samples)}")
print(f"filled within the window  : {len(ttfs)} ({len(ttfs)/max(len(samples),1):.0%})")
if ttfs:
    print(f"median time-to-fill       : {np.median(ttfs):.2f} s")
print("\nMost passive orders at the touch do NOT fill quickly — queue position is real friction.")

In [ ]:
# Adverse selection vs latency, on real data: try to cancel 0.2s after posting, vary how long it takes.
print("cancel decided 0.2s after posting; filled-anyway = adversely selected\n")
for lat_ms in [10, 100, 1000, 5000]:
    filled = 0
    for i, bb in samples:
        r = simulate_passive_order(events, side=0, price=bb, enter_idx=i, size=0.02,
                                   cancel_after_s=0.2, latency_us=lat_ms * 1000)
        filled += int(r.filled)
    print(f"  latency {lat_ms:>5} ms  ->  adverse fills: {filled:>2}/{len(samples)}")

Even in a short, sparse sample the direction is unmistakable: **slower cancels get adversely
selected more often.** On a live, fast venue this is the difference between a profitable and an
unprofitable maker.

---
## Part 5: Go live (optional)

The same dynamics run continuously on the **Realistic Exchange (L3)** arena. Your limit orders take a
real place in the FIFO queue and only fill when they reach the front — with configurable latency on
your cancels. Connect an agent and climb the ladder:

➡️ **[/compete/arena-l3](https://convexpi.ai/compete/arena-l3)** — and the
**[realistic-exchange explainer](https://convexpi.ai/exchange/realistic)**.

The connection pattern is identical to Mission 2 (`from convexpi.arena.client import RemoteAgent`),
but now your fills depend on queue position and your cancels race real trades.

---
## Challenge

Build a **adverse-selection-aware quoting rule** and test it offline on the real data:

1. Rest a buy at the best bid as before, but *also* watch the touch. If the **best ask** ticks down
   past your price (the market is falling toward you), trigger a cancel.
2. Compare your fill rate and adverse-fill rate against a naive always-rest quote across the sampled
   entry points.
3. There's a real trade-off: cancel too eagerly and you never earn the spread; cancel too late and
   you're picked off. Find a threshold that beats naive on net.

When you've got something, write it up and publish it to **[/projects](https://convexpi.ai/projects)**.